In [ ]:
# AG's News Topic Classification Dataset
# ORIGIN
# AG is a collection of more than 1 million news articles. News articles have been gathered from more than 2000 news sources by ComeToMyHead 
# in more than 1 year of activity. ComeToMyHead is an academic news search engine which has been running since July, 2004. The dataset is 
# provided by the academic comunity for research purposes in data mining (clustering, classification, etc), information retrieval (ranking, 
# search, etc), xml, data compression, data streaming, and any other non-commercial activity. For more information, please refer to the link 
# http://www.di.unipi.it/~gulli/AG_corpus_of_news_articles.html .

# The AG's news topic classification dataset is constructed by Xiang Zhang (xiang.zhang@nyu.edu) from the dataset above. It is used as a text 
# classification benchmark in the following paper: Xiang Zhang, Junbo Zhao, Yann LeCun. Character-level Convolutional Networks for Text Classification. 
# Advances in Neural Information Processing Systems 28 (NIPS 2015).

# DESCRIPTION
# The AG's news topic classification dataset is constructed by choosing 4 largest classes from the original corpus. Each class contains 30,000 training
# samples and 1,900 testing samples. The total number of training samples is 120,000 and testing 7,600.

# The file classes.txt contains a list of classes corresponding to each label.

# The files train.csv and test.csv contain all the training samples as comma-sparated values. There are 3 columns in them, corresponding to 
# class index (1 to 4), title and description. The title and description are escaped using double quotes ("), and any internal double quote is
# escaped by 2 double quotes (""). New lines are escaped by a backslash followed with an "n" character, that is "\n".

# About this file
# Add Suggestion
# This file consists of 7600 testing samples of news articles that contain 3 columns. The first column is Class Id, the second column is Title 
# and the third column is Description. The class ids are numbered 1-4 where 1 represents World, 2 represents Sports, 3 represents Business and 4 
# represents Sci/Tech.

In [ ]:
import pandas as pd
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# โหลด AG News Dataset
# สมมุติว่าคุณมีไฟล์ 'ag_news_csv/train.csv' และ 'ag_news_csv/test.csv'
train_df = pd.read_csv('ag_news_csv/train.csv')
test_df = pd.read_csv('ag_news_csv/test.csv')

# ตรวจสอบข้อมูล
print(train_df.head())

# แยกข้อมูลออกเป็นข้อความและป้ายกำกับ
x_train = train_df['Description'].values
y_train = train_df['Class Index'].values
x_test = test_df['Description'].values
y_test = test_df['Class Index'].values

# AG News labels are 1-4; shift to 0-3 for sparse_categorical_crossentropy
y_train = y_train - 1
y_test  = y_test  - 1

# กำหนดจำนวนคำสูงสุดและความยาวสูงสุด
max_words = 10000
max_len = 100  # AG News มีข้อความที่สั้นกว่า IMDB

# Tokenize ข้อความ
tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(x_train)

# แปลงข้อความเป็นลำดับของตัวเลข
x_train = tokenizer.texts_to_sequences(x_train)
x_test = tokenizer.texts_to_sequences(x_test)

# Pad sequences ให้มีความยาวเท่ากัน
x_train = pad_sequences(x_train, maxlen=max_len)
x_test = pad_sequences(x_test, maxlen=max_len)

print(f'ตัวอย่างข้อความ (tokenized): {x_train[0]}')
print(f'Label: {y_train[0]}')

In [ ]:
import pandas as pd
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# โหลด AG News Dataset
train_df = pd.read_csv('ag_news_csv/train.csv')
test_df = pd.read_csv('ag_news_csv/test.csv')

# รวมข้อมูลจากคอลัมน์ "Title" และ "Description"
train_df['combined_text'] = train_df['Title'] + ' ' + train_df['Description']
test_df['combined_text'] = test_df['Title'] + ' ' + test_df['Description']

# ตรวจสอบข้อมูลที่รวมกัน
print(train_df[['Title', 'Description', 'combined_text']].head())

# แยกข้อมูลออกเป็นข้อความและป้ายกำกับ
x_train = train_df['combined_text'].values
y_train = train_df['Class Index'].values
x_test = test_df['combined_text'].values
y_test = test_df['Class Index'].values

# AG News labels are 1-4; shift to 0-3 for sparse_categorical_crossentropy
y_train = y_train - 1
y_test  = y_test  - 1

# กำหนดจำนวนคำสูงสุดและความยาวสูงสุด
max_words = 10000
max_len = 100  # AG News มีข้อความที่สั้นกว่า IMDB

# Tokenize ข้อความ
tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(x_train)

# แปลงข้อความเป็นลำดับของตัวเลข
x_train = tokenizer.texts_to_sequences(x_train)
x_test = tokenizer.texts_to_sequences(x_test)

# Pad sequences ให้มีความยาวเท่ากัน
x_train = pad_sequences(x_train, maxlen=max_len)
x_test = pad_sequences(x_test, maxlen=max_len)

# ตรวจสอบผลลัพธ์
print(f'ตัวอย่างข้อความ (tokenized): {x_train[0]}')
print(f'Label: {y_train[0]}')


In [ ]:
import tensorflow as tf
import keras_tuner as kt

# สร้างโมเดล RNN
def build_rnn_model(hp):
    model = tf.keras.models.Sequential()
    model.add(tf.keras.layers.Embedding(input_dim=max_words, output_dim=hp.Int('embedding_dim', 32, 128, step=32)))
    model.add(tf.keras.layers.LSTM(hp.Int('units', 32, 128, step=32)))
    model.add(tf.keras.layers.Dense(4, activation='softmax'))
    
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# ใช้ Grid Search ผ่าน KerasTuner
tuner_rnn = kt.GridSearch(
    build_rnn_model,
    objective='val_accuracy',
    max_trials=5,  # กำหนดจำนวน trials (จำนวนการทดสอบที่แตกต่างกัน)
    directory='grid_search', 
    project_name='imdb_rnn'
)

# รันการค้นหา
tuner_rnn.search(x_train, y_train, epochs=5, validation_data=(x_test, y_test))

# แสดงไฮเปอร์พารามิเตอร์ที่ดีที่สุด
best_hps_rnn = tuner_rnn.get_best_hyperparameters(num_trials=1)[0]

print(f'Best embedding dimension: {best_hps_rnn.get("embedding_dim")}')
print(f'Best number of LSTM units: {best_hps_rnn.get("units")}')

# สร้างและฝึกโมเดลที่ดีที่สุดด้วยไฮเปอร์พารามิเตอร์ที่ดีที่สุด
best_rnn_model = tuner_rnn.hypermodel.build(best_hps_rnn)
best_rnn_model.fit(x_train, y_train, epochs=5, validation_data=(x_test, y_test))

In [ ]:
# RNN Model: Confusion Matrix
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import seaborn as sns
import matplotlib.pyplot as plt

# # ใช้โมเดล best_rnn_model ที่ได้จากการค้นหาไฮเปอร์พารามิเตอร์ที่ดีที่สุด
# best_rnn_model = tuner_rnn.hypermodel.build(best_hps_rnn)

# # ฝึกโมเดลที่ดีที่สุด
# best_rnn_model.fit(x_train, y_train, epochs=5, validation_data=(x_test, y_test))

# ทำนายผลลัพธ์จาก x_test
y_pred_rnn = best_rnn_model.predict(x_test).argmax(axis=1)

# สร้าง Confusion Matrix
cm_rnn = confusion_matrix(y_test, y_pred_rnn)

# แสดงผล Confusion Matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm_rnn, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

# คำนวณค่า Accuracy, Precision, Recall, และ F1-score
accuracy_rnn = accuracy_score(y_test, y_pred_rnn)
precision_rnn = precision_score(y_test, y_pred_rnn, average='weighted')
recall_rnn = recall_score(y_test, y_pred_rnn, average='weighted')
f1_rnn = f1_score(y_test, y_pred_rnn, average='weighted')

# แสดงผลค่า Accuracy, Precision, Recall, และ F1-score
print(f'Accuracy: {accuracy_rnn:.4f}')
print(f'Precision: {precision_rnn:.4f}')
print(f'Recall: {recall_rnn:.4f}')
print(f'F1-score: {f1_rnn:.4f}')

In [ ]:
# RNN Model with Random Search
# ใช้ KerasTuner สำหรับการปรับแต่งไฮเปอร์พารามิเตอร์
import tensorflow as tf
import keras_tuner as kt

# สร้างโมเดล RNN
def build_rnn_model(hp):
    model = tf.keras.models.Sequential()
    model.add(tf.keras.layers.Embedding(input_dim=max_words, output_dim=hp.Int('embedding_dim', 32, 128, step=32)))
    model.add(tf.keras.layers.LSTM(hp.Int('units', 32, 128, step=32)))
    model.add(tf.keras.layers.Dense(4, activation='softmax'))
    
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# ใช้ Random Search ผ่าน KerasTuner
tuner_rnn = kt.RandomSearch(
    build_rnn_model,
    objective='val_accuracy',
    max_trials=5, 
    directory='random_search', 
    project_name='imdb_rnn'
)

# รันการค้นหา
tuner_rnn.search(x_train, y_train, epochs=5, validation_data=(x_test, y_test))

# แสดงไฮเปอร์พารามิเตอร์ที่ดีที่สุด
best_hps_rnn = tuner_rnn.get_best_hyperparameters(num_trials=1)[0]

print(f'Best embedding dimension: {best_hps_rnn.get("embedding_dim")}')
print(f'Best number of LSTM units: {best_hps_rnn.get("units")}')

# สร้างและฝึกโมเดลที่ดีที่สุดด้วยไฮเปอร์พารามิเตอร์ที่ดีที่สุด
best_rnn_model = tuner_rnn.hypermodel.build(best_hps_rnn)
best_rnn_model.fit(x_train, y_train, epochs=5, validation_data=(x_test, y_test))

In [ ]:
# RNN Model: Confusion Matrix
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import seaborn as sns
import matplotlib.pyplot as plt

# # ใช้โมเดล best_rnn_model ที่ได้จากการค้นหาไฮเปอร์พารามิเตอร์ที่ดีที่สุด
# best_rnn_model = tuner_rnn.hypermodel.build(best_hps_rnn)

# # ฝึกโมเดลที่ดีที่สุด
# best_rnn_model.fit(x_train, y_train, epochs=5, validation_data=(x_test, y_test))

# ทำนายผลลัพธ์จาก x_test
y_pred_rnn = best_rnn_model.predict(x_test).argmax(axis=1)

# สร้าง Confusion Matrix
cm_rnn = confusion_matrix(y_test, y_pred_rnn)

# แสดงผล Confusion Matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm_rnn, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

# คำนวณค่า Accuracy, Precision, Recall, และ F1-score
accuracy_rnn = accuracy_score(y_test, y_pred_rnn)
precision_rnn = precision_score(y_test, y_pred_rnn, average='weighted')
recall_rnn = recall_score(y_test, y_pred_rnn, average='weighted')
f1_rnn = f1_score(y_test, y_pred_rnn, average='weighted')

# แสดงผลค่า Accuracy, Precision, Recall, และ F1-score
print(f'Accuracy: {accuracy_rnn:.4f}')
print(f'Precision: {precision_rnn:.4f}')
print(f'Recall: {recall_rnn:.4f}')
print(f'F1-score: {f1_rnn:.4f}')